In [7]:
import pandas as pd
import numpy as np
import datetime as dt

customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
support_tickets = pd.read_csv('support_tickets.csv')

# Defining the snapshot date
SNAPSHOT_DATE = pd.to_datetime('2025-09-30')

# Filtering orders to prevent data leakage (using only pre-snapshot data)
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_pre_snapshot = orders[orders['order_date'] <= SNAPSHOT_DATE].copy()

# Calculate Recency (days since last order), Frequency (count of orders), Monetary (sum of spend)
rfm = orders_pre_snapshot.groupby('customer_id').agg({
    'order_date': lambda x: (SNAPSHOT_DATE - x.max()).days, # Recency
    'order_id': 'nunique',                                  # Frequency
    'gross_amount': 'sum'                                   # Monetary
}).reset_index()

rfm.rename(columns={
    'order_date': 'Recency',
    'order_id': 'Frequency',
    'gross_amount': 'Monetary'
}, inplace=True)

# Add Non-RFM Signals
# Signal 1: Support Ticket Count
ticket_counts = support_tickets.groupby('customer_id')['ticket_id'].nunique().reset_index()
ticket_counts.rename(columns={'ticket_id': 'support_ticket_count'}, inplace=True)

# Merging back to RFM table
rfm = rfm.merge(ticket_counts, on='customer_id', how='left')

# FIXED WARNING: Instead of inplace=True, we reassign the column
rfm['support_ticket_count'] = rfm['support_ticket_count'].fillna(0)

# Data-Backed Segmentation Logic

def assign_segment(row):
    # DORMANT: EDA showed >129 days recency has an 89.2% churn risk.
    if row['Recency'] > 129:
        return 'Dormant (High Risk)'

    # ONE-TIME BUYERS: EDA showed 1 purchase has a 53.1% churn risk.
    elif row['Frequency'] == 1:
        return 'One-Time Buyer (Needs Onboarding)'

    # CHAMPIONS: EDA showed <= 25 days recency (9.7% churn) and 3+ purchases (16.1% churn) are highly safe.
    elif row['Recency'] <= 25 and row['Frequency'] >= 3:
        return 'Champions (Safe)'

    # AT RISK (SUPPORT ISSUES): They buy frequently, but have logged multiple support tickets recently.
    elif row['support_ticket_count'] >= 2 and row['Monetary'] > 1000:
        return 'High-Value Unhappy'

    # EVERYONE ELSE
    else:
        return 'Regular Active'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Output
output_cols = ['customer_id', 'Segment', 'Recency', 'Frequency', 'Monetary', 'support_ticket_count']
rfm[output_cols].to_csv('segments.csv', index=False)

print("RFM logic complete based on Part 1 EDA thresholds. Segments saved to segments.csv")
print("\nSegment Value Counts:")
print(rfm['Segment'].value_counts())

RFM logic complete based on Part 1 EDA thresholds. Segments saved to segments.csv

Segment Value Counts:
Segment
Regular Active                       718
Dormant (High Risk)                  598
One-Time Buyer (Needs Onboarding)    524
Champions (Safe)                     306
High-Value Unhappy                   254
Name: count, dtype: int64
